# 04. One-stage anchor-based: RetinaNet

1. **Anchors from the data:** how well anchor sets cover the training boxes, by object size.
2. **Sweep on fold 0:** the default model and three experiments on anchors, IoU thresholds and the focal loss.
3. **Full cross-validation** of the best experiment; comparison with the tuned Faster R-CNN (03).
4. **One stage vs two stages:** what removing the proposal stage costs and how RetinaNet compensates.

## Setup

In [ ]:
import subprocess
import sys
from pathlib import Path

if Path("../src/core").exists():
    sys.path.insert(0, "../src")
else:
    repo = Path("/tmp/repo")  # outside /kaggle/working
    if not repo.exists():
        subprocess.run(["git", "clone", "-q", "--depth", "1",
                        "https://github.com/mohinska/ai_hw2_MultipleObjectVideoDetection", str(repo)], check=True)
    sys.path.insert(0, str(repo / "src"))

In [ ]:
!pip install -q faster-coco-eval

In [ ]:
import numpy as np
import pandas as pd

from core import PROTOCOL
from core.cv import compare, cv_table, environment, run_cv
from core.data import assign_folds, find_data_root, load_pool, pool_targets
from core.fusion import fold_ensemble
from core.submission import write_submission
from core.tv import RETINA_SIZES, anchor_match_stats, make_tv_fns, plot_histories

WORK = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("../outputs")
WORK.mkdir(parents=True, exist_ok=True)
environment()

## Data and folds

In [ ]:
DATA = find_data_root()
frames, gt = load_pool(DATA)
targets = pool_targets(gt)
frames["fold"] = frames["video"].map(assign_folds(frames))
frames.groupby("fold").agg(clips=("video", "nunique"), frames=("uid", "size"))

## Anchors grounded in the object sizes
Computed on the training folds of fold 0 only. Aspect ratios (h/w) are taken at the 1/6, 1/2 and 5/6
quantiles of the boxes. For each anchor set and positive IoU threshold: the share of boxes that no anchor
matches (they get only one low-quality match during training), by object size at 1280 px.

In [ ]:
train0 = frames[frames["fold"] != 0]
wh = np.concatenate([targets[u][0][:, 2:] - targets[u][0][:, :2] for u in train0["uid"]])
DATA_RATIOS = tuple(np.round(np.exp(np.quantile(np.log(wh[:, 1] / wh[:, 0]), [1 / 6, 1 / 2, 5 / 6])), 2).tolist())
print("aspect ratios h/w from the data:", DATA_RATIOS)

ANCHOR_SETS = {
    "32-512, 0.5/1/2 (default)": (RETINA_SIZES, (0.5, 1.0, 2.0)),
    "16-256, 0.5/1/2": ((16, 32, 64, 128, 256), (0.5, 1.0, 2.0)),
    f"16-256, {DATA_RATIOS}": ((16, 32, 64, 128, 256), DATA_RATIOS),
    f"12-192, {DATA_RATIOS}": ((12, 24, 48, 96, 192), DATA_RATIOS),
}
coverage, neg_per_pos = {}, {}
for name, (sizes, ratios) in ANCHOR_SETS.items():
    for fg in [0.5, 0.4]:
        t = anchor_match_stats(train0, targets, sizes, ratios, PROTOCOL["imgsz"], fg)
        coverage[(name, fg)] = t["no_positive_anchor"]
        neg_per_pos[(name, fg)] = t.attrs["negative_to_positive"]
table = pd.DataFrame(coverage).T
table.index.names = ["anchors", "positive IoU"]
table["negatives per positive anchor"] = pd.Series(neg_per_pos)
table.round(3)

## Training setup
Same protocol as Faster R-CNN: RetinaNet ResNet-50-FPN v2, COCO-pretrained, new 8-class head, long side 1280 px,
every 5th frame, ignored regions masked, horizontal flip, SGD with warm-up + cosine, 8 epochs, last epoch kept.
lr 0.0025 (the torchvision reference 0.01 at batch 16, scaled to batch 4).

The aspect ratios measured from the boxes (printed above; 0.56 / 1.06 / 1.96 on the train split in 01) are close to
the default 0.5 / 1 / 2, so the ratios are kept and only the anchor scale is tuned.

Each experiment adds one change to the previous one, starting from the anchor table above:
- **Smaller anchors, 16–256 px:** most objects are 16–64 px at 1280, below the default smallest anchor.
- **IoU thresholds 0.4 / 0.3** (default 0.5 / 0.4): more anchors count as positive for small boxes.
- **No focal term, γ = 0** (loss modification, on the default model): plain cross-entropy shows what the
  focal loss fixes.

In [ ]:
BASE = dict(arch="retinanet", lr=0.0025)
SMALL = dict(anchor_sizes=(16, 32, 64, 128, 256))
EXPERIMENTS = {
    "retina_default": BASE,
    "retina_anchors": {**BASE, **SMALL},
    "retina_anchors_iou": {**BASE, **SMALL, "fg_iou": 0.4, "bg_iou": 0.3},
    "retina_no_focal": {**BASE, "focal_gamma": 0.0},
}
FRCNN = ["frcnn_default", "frcnn_roi_pos50", "frcnn_anchors", "frcnn_boxloss2", "frcnn_step_lr"]  # from 03


def config(name):
    return {**PROTOCOL, **EXPERIMENTS[name], "name": name}

## Sweep on fold 0

In [ ]:
for name in EXPERIMENTS:
    print(name, EXPERIMENTS[name])
    run_cv(frames, gt, config(name), *make_tv_fns(config(name), WORK, gt, targets), WORK, folds=[0])

### Fold-0 comparison
Initial vs tuned RetinaNet, the Faster R-CNN runs of 03 and the YOLO baseline on the same held-out fold
(03 and 02 outputs attached as input).

In [ ]:
sweep = compare([*EXPERIMENTS, *FRCNN, "baseline"], WORK, fold=0)
display(sweep.round(4))
plot_histories(EXPERIMENTS, 0, WORK);

## Full cross-validation of the best experiment
It is chosen on fold 0, so its fold-0 score is slightly optimistic.

In [ ]:
BEST = sweep.loc[sweep.index.isin(EXPERIMENTS), "mAP"].idxmax()
print("best on fold 0:", BEST)
result = run_cv(frames, gt, config(BEST), *make_tv_fns(config(BEST), WORK, gt, targets), WORK)

In [ ]:
display(compare([BEST, *FRCNN, "baseline"], WORK).round(4))  # out-of-fold over all frames
cv_table(result).round(4)

## One stage vs two stages
Removing the proposal stage causes three problems:
- **Foreground-background imbalance.** The classifier scores every anchor (~10⁵ per frame, see the anchor table),
  and only a few hundred match an object. Easy background dominates the loss. RetinaNet answers with the focal loss,
  which down-weights well-classified examples, and a prior bias on the classification layer
  (`retina_no_focal` shows the effect).
- **No second look at the box.** Faster R-CNN re-pools features for each proposal and refines the box; RetinaNet
  regresses each anchor once, so localisation (AP75) tends to be weaker.
- **Anchor dependence.** An object that no anchor overlaps enough gets only one low-quality match and is
  under-trained, which hits small objects first. Smaller anchors and lower IoU thresholds address it;
  learned assignment (ATSS) or anchor-free heads (05) remove it.

*(After the run: confirm or refute with the tables above: AP75 and AP_small vs Faster R-CNN, focal vs no focal.)*

## Submission
The four fold models of the best experiment, fused by Weighted Boxes Fusion.

In [ ]:
test = pd.read_csv(DATA / "test_images.csv")
_, predict_fn = make_tv_fns(config(BEST), WORK, gt, targets)
test_dets = fold_ensemble(BEST, predict_fn, [str(DATA / f) for f in test["file_name"]], WORK)
write_submission(test_dets, test, WORK / f"submission_{BEST}.csv").head()